In [1]:
!pip install -q transformers librosa scipy huggingface_hub

In [2]:
import os
import glob
import numpy as np
import librosa
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import HubertModel, Wav2Vec2FeatureExtractor
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()

mhubertmodel_path = "utter-project/mHuBERT-147"
processor = Wav2Vec2FeatureExtractor.from_pretrained(mhubertmodel_path)
audio_encoder = HubertModel.from_pretrained(mhubertmodel_path, attn_implementation="eager").to(device)

if num_gpus > 1:
    audio_encoder = torch.nn.DataParallel(audio_encoder)

audio_encoder.eval()
for param in audio_encoder.parameters():
    param.requires_grad = False

preprocessor_config.json:   0%|          | 0.00/227 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

In [3]:
VARIANT = "eng"
output_base_dir = "/tmp/mhubert_12layers"
BATCH_SIZE = 16

def format_mhubert_features(hidden_states, gt_frame_num):
    if hidden_states.shape[1] % 2 != 0:
        hidden_states = hidden_states[:, :-1, :]
        
    target_audio_len = gt_frame_num * 2
    current_audio_len = hidden_states.shape[1]
    
    if current_audio_len < target_audio_len:
        hidden_states = hidden_states.transpose(1, 2)
        hidden_states = F.interpolate(hidden_states, size=target_audio_len, align_corners=True, mode='linear')
        hidden_states = hidden_states.transpose(1, 2)
        
    elif current_audio_len > target_audio_len:
        hidden_states = hidden_states[:, :target_audio_len, :]
        
    return hidden_states

def find_vertices_file(base_filename, vertices_dirs):
    for v_dir in vertices_dirs:
        target_path = os.path.join(v_dir, f"{base_filename}_vertices.pt")
        if os.path.exists(target_path):
            return target_path
    return None

class MHubertDataset(Dataset):
    def __init__(self, file_list):
        self.file_list = file_list

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        wav_path, vertices_path, lang, out_dir = self.file_list[idx]
        base_name = os.path.splitext(os.path.basename(wav_path))[0]
        
        speech_array, _ = librosa.load(wav_path, sr=16000)
        vertices = torch.load(vertices_path, map_location='cpu')
        gt_frame_num = vertices.shape[0]
        
        return {
            "speech_array": speech_array,
            "gt_frame_num": gt_frame_num,
            "base_name": base_name,
            "out_dir": out_dir
        }

def collate_fn(batch):
    speech_arrays = [item["speech_array"] for item in batch]
    gt_frame_nums = [item["gt_frame_num"] for item in batch]
    base_names = [item["base_name"] for item in batch]
    out_dirs = [item["out_dir"] for item in batch]
    
    audio_inputs = processor(
        speech_arrays, 
        sampling_rate=16000, 
        return_tensors="pt", 
        padding=True,
        return_attention_mask=True
    )
    
    return audio_inputs.input_values, audio_inputs.attention_mask, gt_frame_nums, base_names, out_dirs


In [4]:
dataset_mapping = {
    "ara": (
        "/kaggle/input/datasets/jorx04/fully-collected-dataset/audio/ara",
        [
            "/kaggle/input/datasets/jorx04/ara-vertices-p1/vertices",
            "/kaggle/input/datasets/jorx04/ara-vertices-p2/vertices"
        ]
    ),
    "eng": (
        "/kaggle/input/datasets/jorx04/fully-collected-dataset/audio/eng",
        [
            "/kaggle/input/datasets/jorx04/eng-vertices-p1/vertices",
            "/kaggle/input/datasets/jorx04/eng-vertices-p2/vertices"
        ]
    )
}

target_lang = VARIANT
audio_dir, vertices_dirs = dataset_mapping[target_lang]
out_dir = os.path.join(output_base_dir, target_lang)
os.makedirs(out_dir, exist_ok=True)

all_valid_files = []
audio_files = glob.glob(os.path.join(audio_dir, '*.wav'))

for audio_path in audio_files:
    base_name = os.path.splitext(os.path.basename(audio_path))[0]
    save_path = os.path.join(out_dir, f"{base_name}.pt")
    
    if os.path.exists(save_path):
        continue
        
    vertices_path = find_vertices_file(base_name, vertices_dirs)
    if vertices_path:
        all_valid_files.append((audio_path, vertices_path, target_lang, out_dir))

all_valid_files.sort(key=lambda x: os.path.getsize(x[0]))

print(f"Running Variant: {VARIANT.upper()} | Files to process: {len(all_valid_files)}")

dataset = MHubertDataset(all_valid_files)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)

with torch.no_grad():
    for input_values, attention_mask, gt_frame_nums, base_names, out_dirs in tqdm(dataloader, desc="Processing"):
        input_values = input_values.to(device)
        attention_mask = attention_mask.to(device)
        
        outputs = audio_encoder(
            input_values, 
            attention_mask=attention_mask, 
            output_hidden_states=True
        )
        
        stacked_hidden = torch.stack(outputs.hidden_states[1:], dim=1)
        
        base_model = audio_encoder.module if hasattr(audio_encoder, 'module') else audio_encoder
        unpadded_lengths = base_model._get_feat_extract_output_lengths(attention_mask.sum(-1))
        
        for i in range(len(base_names)):
            valid_len = unpadded_lengths[i].item()
            valid_hidden = stacked_hidden[i, :, :valid_len, :]
            
            aligned_features = format_mhubert_features(valid_hidden, gt_frame_nums[i])
            
            save_path = os.path.join(out_dirs[i], f"{base_names[i]}.pt")
            torch.save(aligned_features.half().cpu(), save_path)
            
        del input_values, attention_mask, outputs, stacked_hidden
        torch.cuda.empty_cache()

Running Variant: ENG | Files to process: 11055


Processing:   0%|          | 0/691 [00:00<?, ?it/s]

In [5]:
import os
import glob
import torch
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

def audit_single_file(p_path, vertices_dirs):
    base_name = os.path.splitext(os.path.basename(p_path))[0]
    gt_path = None
    
    for v_dir in vertices_dirs:
        temp_path = os.path.join(v_dir, f"{base_name}_vertices.pt")
        if os.path.exists(temp_path):
            gt_path = temp_path
            break
            
    if not gt_path:
        return {"status": "missing_gt", "base_name": base_name}
        
    processed_data = torch.load(p_path, map_location='cpu')
    gt_data = torch.load(gt_path, map_location='cpu')
    
    has_nan = torch.isnan(processed_data).any().item()
    
    has_shape_error = False
    if len(processed_data.shape) != 3 or processed_data.shape[0] != 12 or processed_data.shape[2] != 768:
        has_shape_error = True
        t_processed = -1
    else:
        t_processed = processed_data.shape[1]
        
    t_gt = gt_data.shape[0]
    
    is_mismatch = (t_processed != (t_gt * 2)) and not has_shape_error
    mismatch_str = f"{base_name}: Proc({t_processed}) vs GT({t_gt})" if is_mismatch else ""
    
    del processed_data, gt_data
    
    return {
        "status": "ok",
        "base_name": base_name,
        "has_nan": has_nan,
        "has_shape_error": has_shape_error,
        "is_mismatch": is_mismatch,
        "mismatch_str": mismatch_str
    }

def run_dataset_audit(dataset_mapping, processed_base_dir, num_workers=8):
    report = {}

    for lang, (audio_dir, vertices_dirs) in dataset_mapping.items():
        audio_files = glob.glob(os.path.join(audio_dir, '*.wav'))
        processed_files = glob.glob(os.path.join(processed_base_dir, lang, '*.pt'))
        
        lang_report = {
            "audio_count": len(audio_files),
            "processed_count": len(processed_files),
            "mismatches": [],
            "length_errors": 0,
            "nan_errors": [],
            "shape_errors": [],
            "missing_gt": 0
        }

        print(f"\nAuditing {lang.upper()} dataset...")
        
        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            futures = {executor.submit(audit_single_file, p_path, vertices_dirs): p_path for p_path in processed_files}
            
            for future in tqdm(as_completed(futures), total=len(processed_files), desc=f"Checking {lang}"):
                result = future.result()
                
                if result["status"] == "missing_gt":
                    lang_report["missing_gt"] += 1
                    continue
                    
                if result["has_nan"]:
                    lang_report["nan_errors"].append(result["base_name"])
                    
                if result["has_shape_error"]:
                    lang_report["shape_errors"].append(result["base_name"])
                    
                if result["is_mismatch"]:
                    lang_report["length_errors"] += 1
                    lang_report["mismatches"].append(result["mismatch_str"])

        report[lang] = lang_report

    print("\n--- DATASET AUDIT REPORT ---")
    for lang, data in report.items():
        print(f"\nLANGUAGE: {lang.upper()}")
        print(f"  - Original Audio Files: {data['audio_count']}")
        print(f"  - Processed Tensors: {data['processed_count']}")
        print(f"  - Files Missing GT: {data['missing_gt']}")
        
        print(f"  - NaN Errors: {len(data['nan_errors'])}")
        if data['nan_errors']:
            print(f"    First 3 NaNs: {data['nan_errors'][:3]}")
            
        print(f"  - Shape Architecture Errors: {len(data['shape_errors'])}")
        if data['shape_errors']:
            print(f"    First 3 Shape Errors: {data['shape_errors'][:3]}")
            
        print(f"  - Length Alignment Errors: {data['length_errors']}")
        if data['length_errors'] > 0:
            print(f"    First 3 length errors: {data['mismatches'][:3]}")

run_dataset_audit(dataset_mapping, "/tmp/mhubert_12layers", num_workers=8)


Auditing ARA dataset...


Checking ara: 0it [00:00, ?it/s]


Auditing ENG dataset...


Checking eng:   0%|          | 0/11055 [00:00<?, ?it/s]


--- DATASET AUDIT REPORT ---

LANGUAGE: ARA
  - Original Audio Files: 8527
  - Processed Tensors: 0
  - Files Missing GT: 0
  - NaN Errors: 0
  - Shape Architecture Errors: 0
  - Length Alignment Errors: 0

LANGUAGE: ENG
  - Original Audio Files: 11055
  - Processed Tensors: 11055
  - Files Missing GT: 0
  - NaN Errors: 0
  - Shape Architecture Errors: 0
  - Length Alignment Errors: 0


In [7]:
import os
import json
import zipfile
import glob
from tqdm.auto import tqdm
from kaggle_secrets import UserSecretsClient

files_to_zip = glob.glob(f"{output_base_dir}/**/*.pt", recursive=True)
upload_dir = f"/tmp/upload_{VARIANT}"
output_zip = f"{upload_dir}/mhubert_{VARIANT}.zip"

os.makedirs(upload_dir, exist_ok=True)
with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_STORED) as zipf:
    for file in tqdm(files_to_zip, desc="Zipping"):
        arcname = os.path.relpath(file, output_base_dir)
        zipf.write(file, arcname)
        os.remove(file)

user_secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = user_secrets.get_secret("KAGGLE_USERNAME")
os.environ['KAGGLE_KEY'] = user_secrets.get_secret("KAGGLE_KEY")

!kaggle datasets init -p {upload_dir}

metadata_path = os.path.join(upload_dir, "dataset-metadata.json")
with open(metadata_path, 'r') as f:
    meta = json.load(f)

dataset_slug = f"mhubert-147-features-{VARIANT}"
meta['id'] = f"{os.environ['KAGGLE_USERNAME']}/{dataset_slug}"
meta['title'] = f"mHuBERT-147 12-Layer Features ({VARIANT.upper()})"

with open(metadata_path, 'w') as f:
    json.dump(meta, f, indent=4)

!kaggle datasets create -p {upload_dir}

Zipping:   0%|          | 0/11055 [00:00<?, ?it/s]

Data package template written to: /tmp/upload_eng/dataset-metadata.json
Starting upload for file mhubert_eng.zip
100%|██████████████████████████████████████| 47.8G/47.8G [09:21<00:00, 91.4MB/s]
Upload successful: mhubert_eng.zip (48GB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/jorx04/mhubert-147-features-eng


In [8]:
!du -sh /tmp/* /kaggle/working/*

4.0K	/tmp/clean-layer.sh
460K	/tmp/mhubert_12layers
4.0K	/tmp/pymp-s8tr_nfj
4.0K	/tmp/pymp-wu7r1pn3
4.0K	/tmp/torchinductor_root
48G	/tmp/upload_eng
0	/tmp/uv-05f7b0856e340464.lock
0	/tmp/uv-setuptools-0833c489c6f2b4e3.lock
0	/tmp/uv-setuptools-17caaeb94204b78b.lock
0	/tmp/uv-setuptools-2384f4f145979e25.lock
0	/tmp/uv-setuptools-5b308e0b745393cb.lock
0	/tmp/uv-setuptools-5eea02bf76b319db.lock
0	/tmp/uv-setuptools-682bc7d72f484173.lock
0	/tmp/uv-setuptools-7ed6e4c56c5251fe.lock
0	/tmp/uv-setuptools-812fcd445785e3eb.lock
0	/tmp/uv-setuptools-85238c1fc5c14533.lock
0	/tmp/uv-setuptools-871b39a28910251c.lock
0	/tmp/uv-setuptools-905073b52d85b557.lock
0	/tmp/uv-setuptools-95f0f79d35ee896f.lock
0	/tmp/uv-setuptools-9ba9863f0ddbaad4.lock
0	/tmp/uv-setuptools-a6deb615a522d5ec.lock
0	/tmp/uv-setuptools-a84f10b927b58cc7.lock
0	/tmp/uv-setuptools-bc86c5dd5d473b56.lock
0	/tmp/uv-setuptools-c97b7e4e1efa8cc8.lock
0	/tmp/uv-setuptools-d697db28824a92db.lock
0	/tmp/uv-setuptools-e1656a64e3d74a22.lock
0	